# MicroPlant — Model Training

This notebook trains the MicroPlant model for plant disease classification.

We explore:
- Baseline training (MicroPlant)
- Teacher model (ResNet18)
- Knowledge Distillation (KD)

The goal is to build an accurate yet lightweight model suitable for edge deployment.

In [11]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt

from src.preprocessing import get_dataloaders
from src.architectures import get_microplant, get_teacher_model
from src.training import train_model, validate
from src.utils import count_parameters, count_model_bytes

## Configuration

We define training parameters and device setup.

In [21]:
DATA_DIR = "../data/MicroPlantDataset"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

BATCH_SIZE = 64
EPOCHS = 15
LR = 0.001

print(f"using : {DEVICE}")

using : cpu


## Load Dataset

We use the predefined data pipeline with augmentation and proper splitting.

In [14]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE
)

print("Classes:", class_names)

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']


## Baseline Model (MicroPlant)

We first train the MicroPlant model without any distillation.

This serves as a baseline for comparison.

In [15]:
baseline_model = get_microplant(num_classes=len(class_names)).to(DEVICE)

print("Parameters:", count_parameters(baseline_model))
count_model_bytes(baseline_model)

Parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

In [29]:
baseline_model = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    save_name="../models/microplant_baseline",
    device=DEVICE
)

Training:   0%|          | 0/41 [00:00<?, ?it/s]


AssertionError: Torch not compiled with CUDA enabled